In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm

**SUMSTATS HARMONIZATION**

**BBJ**

In [ ]:
import pandas as pd

# Step 1: Load BBJ variants
bbj = pd.read_csv('/scratch/07880/devansh/CAD/New_PRS/bbj.CAD.rsq07.mac10/CAD.auto.rsq07.mac10.txt.gz', delim_whitespace = True, compression="gzip")
split = bbj["SNPID"].str.split("_", expand=True)
bbj["CHR"] = split[0].astype(str)
bbj["POS"] = split[1].astype(int)
bbj["REF"] = split[2]
bbj["ALT"] = split[3]

# Optional: reduce BBJ to unique variants for faster matching
bbj_variants = bbj[["CHR", "POS", "REF", "ALT"]].drop_duplicates()

In [ ]:
bbj_variants.tail()

In [ ]:
# Step 2: Prepare output list
matched_chunks = []

# Step 3: Stream and match in chunks
chunksize = 500000
reader = pd.read_csv("dbsnp_rsmap.tsv.gz", sep="\t", names=["CHR_raw", "POS", "REF", "ALT", "rsID"], compression="gzip", chunksize=chunksize)

for i, chunk in enumerate(reader):
    # Normalize chromosome: extract number from RefSeq
    chunk["CHR"] = chunk["CHR_raw"].str.extract(r'NC_0+(\d+)\.')[0]
    chunk["CHR"] = chunk["CHR"].astype(str).replace({"23": "X", "24": "Y"})

    # Explode multiallelic ALT
    chunk = chunk.assign(ALT=chunk["ALT"].str.split(",")).explode("ALT")

    # Subset columns
    chunk = chunk[["CHR", "POS", "REF", "ALT", "rsID"]]

    # Merge with BBJ
    matched = pd.merge(bbj_variants, chunk, on=["CHR", "POS", "REF", "ALT"], how="inner")
    matched_chunks.append(matched)

    print(f"✅ Processed chunk {i+1}")

# Step 4: Combine all matched rsIDs
rsid_map = pd.concat(matched_chunks, ignore_index=True)

# Step 5: Merge rsID back into original BBJ data
bbj_with_rsid = pd.merge(bbj, rsid_map, on=["CHR", "POS", "REF", "ALT"], how="left")

# Step 6: Save
bbj_with_rsid.to_csv("bbj_sumstats_with_rsids.txt", sep="\t", index=False)
print("✔️ Matching complete and saved.")

In [ ]:
df = pd.read_csv('bbj_sumstats_with_rsids.txt', sep = '\t')

In [ ]:
df.head()

In [ ]:
missing = df[df["rsID"].isnull()]
print(f"{len(missing)} variants did not map to rsIDs.")

In [ ]:
len(df)

In [ ]:
df = df.dropna()

In [ ]:
df = df.rename(columns={
        "rsID": "SNP",
        "Allele2": "A1",
        "Allele1": "A2",
        "BETA": "BETA",
        "SE": "SE",
        "p.value": "P",
        "N": "N"
})
df = df[["SNP", "A1", "A2", "BETA", "SE", "P", "N"]].dropna()
df.to_csv("harmonized_bbj.txt", sep="\t", index=False)

In [ ]:
df.head()

**FINNGEN**

In [ ]:
df_finn = pd.read_csv('/scratch/07880/devansh/CAD/New_PRS/finngen_R12_I9_CORATHER.gz', delim_whitespace = True, compression="gzip")

In [ ]:
df_finn.head()

In [ ]:
df_finn = df_finn.rename(columns={
    "rsids":'SNP',
        "alt": "A1",
        "ref": "A2",
        "beta": "BETA",
        "sebeta": "SE",
        "pval": "P"
    })

# Set sample size — adjust for your phenotype
df_finn["N"] = 479478


df_finn = df_finn[["SNP", "A1", "A2", "BETA", "SE", "P", "N"]].dropna()
df_finn.to_csv('harmonized_finngen.txt', sep="\t", index=False)

In [ ]:
df_finn.head()

**MVP**

In [ ]:
mvp = pd.read_csv("/scratch/07880/devansh/CAD/New_PRS/MVP_R4.1000G_AGR.GIA.PheCodes_CirculatorySystem_batch1/MVP_R4.1000G_AGR.Phe_411_4.EUR.GIA.dbGaP.txt.gz", delim_whitespace = True, compression="gzip")

In [ ]:
mvp.head()

In [ ]:
mvp['num_samples'].value_counts()

In [ ]:
mvp = mvp.rename(columns={
        "SNP_ID": "SNP",
        "chrom": "CHR",
        "pos": "POS",
        "ea": "A1",
        "ref": "REF",
        "alt": "ALT",
        "or": "OR",
        "pval": "P",
        "num_samples": "N"
    })

# Exclude bad rows
mvp = mvp[mvp["OR"].notnull() & (mvp["OR"] > 0)]

# Infer non-effect allele (A2)
mvp["A2"] = np.where(mvp["A1"] == mvp["REF"], mvp["ALT"], mvp["REF"])

# Compute BETA and SE from OR and P
mvp["BETA"] = np.log(mvp["OR"])
mvp["Z"] = np.sign(mvp["BETA"]) * abs(norm.ppf(mvp["P"].clip(lower=1e-300)))
mvp["SE"] =mvp["BETA"] / mvp["Z"]
mvp = mvp.replace([np.inf, -np.inf], np.nan)


mvp = mvp[["SNP", "A1", "A2", "BETA", "SE", "P", "N"]].dropna()
mvp.to_csv('harmonized_mvp.txt', sep="\t", index=False)

In [ ]:
mvp.head()

In [ ]:
len(mvp)

**UKBB**

In [ ]:
ukb = pd.read_csv('CAD_step2_CAD.regenie', sep=r'\s+', engine='python')
ukb.head()

# Rename for clarity
ukb = ukb.rename(columns={
    "ID":"SNP",
        "ALLELE0": "A2",
        "ALLELE1": "A1",
        "LOG10P": "-log10(P)"
    })

# Recalculate raw P-value from -log10(P)
ukb["P"] = 10 ** (-ukb["-log10(P)"])


ukb = ukb[["SNP", "A1", "A2", "BETA", "SE", "P", "N"]].dropna()
ukb.to_csv('harmonized_ukb.txt', sep="\t", index=False)

In [ ]:
ukb.head()

In [ ]:
# Load all SNP rsID lists
ukb = set(pd.read_csv("harmonized_ukb.txt", sep="\t")["SNP"])
mvp = set(pd.read_csv("harmonized_mvp.txt", sep="\t")["SNP"])
fin = set(pd.read_csv("harmonized_finngen.txt", sep="\t")["SNP"])

# Take intersection
shared = ukb & mvp & fin 
print(f"{len(shared):,} SNPs shared across all 3 studies.")

**META GWAS CHECK**

In [ ]:
import pandas as pd

df = pd.read_csv("meta_CAD_results1.tbl", sep="\t")
df["n_studies"] = df["Direction"].apply(lambda x: len(x.replace("?", "")))

# Count per group
print(df["n_studies"].value_counts())

In [ ]:
# Save SNPs present in 3 or more studies
df_3plus = df[df["n_studies"] >= 3]
df_3plus.to_csv("meta_CAD_results_3plus_studies.txt", sep="\t", index=False)

# Save SNPs present in 2 or more studies
df_2plus = df[df["n_studies"] >= 2]
df_2plus.to_csv("meta_CAD_results_2plus_studies.txt", sep="\t", index=False)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# -----------------------------
# Load METAL meta-analysis file
# -----------------------------
meta_file = "meta_CAD_results1.tbl"
df = pd.read_csv(meta_file, sep="\t")

# Rename rsID column
df = df.rename(columns={"MarkerName": "SNP"})
df['P-value'] = pd.to_numeric(df['P-value'], errors='coerce')

# -----------------------------
# Load dbSNP rsID to CHR/POS map
# -----------------------------
rsmap = pd.read_csv("dbsnp_subset.txt", sep="\t", names=["CHR_raw", "POS", "REF", "ALT", "SNP"])

# Convert RefSeq ID to numeric chromosome
rsmap["CHR"] = rsmap["CHR_raw"].str.extract(r'NC_0+(\d+)\.')[0]
rsmap["CHR"] = rsmap["CHR"].replace({"23": "X", "24": "Y"})
rsmap = rsmap[["SNP", "CHR", "POS"]].drop_duplicates()

print(rsmap.head())

# Merge position info into meta results
df = df.merge(rsmap, on="SNP", how="inner")
df = df[df["CHR"].isin([str(i) for i in range(1, 23)])]  # autosomes only
df["CHR"] = df["CHR"].astype(int)
df["POS"] = df["POS"].astype(int)
df["-log10(P)"] = -np.log10(df["P-value"])

# Sort by chromosome and position
df = df.sort_values(["CHR", "POS"]).reset_index(drop=True)

# -----------------------------
# Manhattan Plot
# -----------------------------
def plot_manhattan(df, title="Manhattan Plot"):
    df["ind"] = range(len(df))
    df_grouped = df.groupby("CHR")

    fig, ax = plt.subplots(figsize=(14, 6))
    colors = ["#4e79a7", "#f28e2c"]
    x_labels = []
    x_labels_pos = []

    for i, (num, group) in enumerate(df_grouped):
        group.plot(kind='scatter', x='ind', y='-log10(P)', color=colors[i % 2], ax=ax, s=8, alpha=0.7)
        x_labels.append(num)
        x_labels_pos.append((group['ind'].iloc[-1] + group['ind'].iloc[0]) // 2)

    ax.set_xticks(x_labels_pos)
    ax.set_xticklabels(x_labels)
    ax.set_xlabel("Chromosome")
    ax.set_ylabel("-log10(P-value)")
    ax.set_title(title)
    ax.axhline(-np.log10(5e-8), color='red', linestyle='--', linewidth=1)
    plt.tight_layout()
    plt.show()

# -----------------------------
# QQ Plot
# -----------------------------
from scipy.stats import chi2

def plot_qq(df, title="QQ Plot"):
    # Drop missing p-values
    pvals = df['P-value'].dropna()
    pvals = pvals[pvals > 0]  # avoid -log(0)

    # Compute observed chi2 stats
    chisq_obs = chi2.isf(pvals, df=1)  # inverse survival function = -log(1-p)

    # λGC = median observed / expected median under null
    lambda_gc = np.median(chisq_obs) / 0.4549364

    # QQ plot values
    expected = -np.log10(np.linspace(1 / len(pvals), 1, len(pvals)))
    observed = -np.log10(pvals.sort_values().values)

    # Plot
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(expected, observed, '.', alpha=0.5)
    ax.plot([0, max(expected)], [0, max(expected)], color='red', linestyle='--')
    ax.set_xlabel('Expected -log10(P)')
    ax.set_ylabel('Observed -log10(P)')
    ax.set_title(f"{title}\nλGC = {lambda_gc:.3f}")
    plt.tight_layout()
    plt.show()
    print(f"✅ QQ plot complete — Genomic inflation factor λGC = {lambda_gc:.3f}")

# -----------------------------
# Run Plots
# -----------------------------
plot_manhattan(df, title="CAD Meta-analysis Manhattan Plot")
plot_qq(df, title="CAD Meta-analysis QQ Plot")

In [ ]:
# !zcat dbsnp_rsmap.tsv.gz | awk 'NR==FNR{keep[$1]; next} ($5 in keep)' rsids_sorted.txt - > dbsnp_subset.txt

In [ ]:
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# from scipy.stats import uniform
# from scipy.stats import chi2
# %matplotlib inline

# # -------- Load METAL output --------
# file = "meta_CAD_results1.tbl"
# df = pd.read_csv(file, sep="\t")

# # -------- Parse CHR and POS from MarkerName --------
# df[['CHR', 'POS', 'REF', 'ALT']] = df['MarkerName'].str.split('_', expand=True)
# df = df[df['CHR'].isin([str(i) for i in range(1, 23)])]
# df['CHR'] = df['CHR'].astype(int)
# df['POS'] = df['POS'].astype(int)
# df['-log10(P)'] = -np.log10(df['P-value'])

# # Sort by chromosome and position
# df = df.sort_values(['CHR', 'POS'])

# # -------- Manhattan Plot --------
# def plot_manhattan(df, title="Manhattan Plot", genomewide_line=True):
#     df['ind'] = range(len(df))
#     df_grouped = df.groupby('CHR')

#     fig, ax = plt.subplots(figsize=(14, 6))
#     colors = ['#4e79a7', '#f28e2c']
#     x_labels = []
#     x_labels_pos = []

#     for i, (num, group) in enumerate(df_grouped):
#         group.plot(kind='scatter', x='ind', y='-log10(P)', color=colors[i % 2], ax=ax, s=8, alpha=0.7)
#         x_labels.append(num)
#         x_labels_pos.append((group['ind'].iloc[-1] + group['ind'].iloc[0]) // 2)

#     ax.set_xticks(x_labels_pos)
#     ax.set_xticklabels(x_labels)
#     ax.set_xlabel('Chromosome')
#     ax.set_ylabel('-log10(P-value)')
#     ax.set_title(title)
#     if genomewide_line:
#         ax.axhline(-np.log10(5e-8), color='red', linestyle='--', linewidth=1)
#     plt.tight_layout()
#     plt.show()
#     plt.close()
#     print("✅ Manhattan plot saved as manhattan_plot.png")

# # -------- QQ Plot --------
# def plot_qq(df, title="QQ Plot"):
#     observed_p = df['P-value'].dropna().sort_values()
#     expected = -np.log10(np.linspace(1/len(observed_p), 1, len(observed_p)))
#     observed = -np.log10(observed_p.values)

#     fig, ax = plt.subplots(figsize=(6, 6))
#     ax.plot(expected, observed, '.', alpha=0.5)
#     ax.plot([0, max(expected)], [0, max(expected)], color='red', linestyle='--')
#     ax.set_xlabel('Expected -log10(P)')
#     ax.set_ylabel('Observed -log10(P)')
#     ax.set_title(title)
#     plt.tight_layout()
#     plt.show()
#     plt.close()
#     print("✅ QQ plot saved as qq_plot.png")

# # -------- Run Plots --------
# plot_manhattan(df)
# plot_qq(df)

In [ ]:
import pandas as pd

df = pd.read_csv("meta_CAD_results_2plus_studies.txt", sep="\t")
df_prs = df.rename(columns={"MarkerName": "SNP", "Allele1": "A1", "Allele2":"A2", "Effect": "BETA", "P-value":'P'})
df_prs = df_prs[["SNP", "A1", "A2", "BETA", "P"]]
df_prs['A1'] = df_prs['A1'].str.upper()
df_prs['A2'] = df_prs['A2'].str.upper()
df_prs.to_csv("PRS_INPUT_FILES/prs_input_2stduy_SNPs_EUR.txt", sep="\t", index=False)

In [ ]:
# GENO_PATH="/corral/utexas/UKB-Imaging-Genetics/UKB_GENOTYPE_QC_400k/merged_maf0.01_biallel_bbf_400k/merged_sub_chrom_maf0.01"
# PHENO_FILE="PRS_INPUT_FILES/subset_ids.txt"  # your file with FID IID CAD
# KEEP_FILE="PRS_INPUT_FILES/keep.txt"
# SUMSTATS="meta_output_for_prs.txt"  # your harmonized summary stats

In [ ]:
# !/work2/07880/devansh/lonestar/software/plink --bfile "$GENO_PATH" --keep "$KEEP_FILE" --make-bed --out PRS_INPUT_FILES/subset_data

In [ ]:
# import pandas as pd

# # Step 1: Load .bim file and build mapping
# bim = pd.read_csv("PRS_INPUT_FILES/subset_data.bim", sep="\t", header=None)
# bim.columns = ["CHR", "RSID", "CM", "BP", "A1", "A2"]

# # Construct position-based keys and flipped versions (to be safe)
# bim["key1"] = bim["CHR"].astype(str) + "_" + bim["BP"].astype(str) + "_" + bim["A1"] + "_" + bim["A2"]
# bim["key2"] = bim["CHR"].astype(str) + "_" + bim["BP"].astype(str) + "_" + bim["A2"] + "_" + bim["A1"]
# key_to_rsid = pd.Series(bim["RSID"].values, index=bim["key1"]).to_dict()
# key_to_rsid.update(pd.Series(bim["RSID"].values, index=bim["key2"]).to_dict())

# # Step 2: Load summary stats
# sumstats = pd.read_csv("meta_CAD_results2.tbl", sep="\t")

# # Step 3: Map MarkerName to RSID
# sumstats["RSID"] = sumstats["MarkerName"].str.upper().map(key_to_rsid)

# # Drop unmapped SNPs
# sumstats = sumstats.dropna(subset=["RSID"])

# # Step 4: Build PRS input file
# prs = sumstats[["RSID", "Allele1", "Allele2" "Effect", "P-value"]].copy()
# prs.columns = ["SNP", "A1", "A2","BETA", "P"]
# prs['A1'] = prs['A1'].str.upper()
# prs.to_csv("PRS_INPUT_FILES/prs_input_keyed.txt", sep="\t", index=False)

# print(f"✅ Created PRS_INPUT_FILES/prs_input_keyed.txt with {len(prs)} variants.")

**PRS CALCULATIONS**

In [ ]:
!pwd

In [ ]:
# # --- Configuration ---
# GENO_PREFIX="PRS_INPUT_FILES/subset_data"
# SUMSTATS="PRS_INPUT_FILES/prs_input_keyed_pre_clump.txt"  # should contain SNP, P, BETA
# PHENO_COV="PRS_INPUT_FILES/subset_ids_plink.txt"       # FID IID CAD PC1 PC2
# OUTPUT_PREFIX="PRS_INPUT_FILES/prs_ct"

In [ ]:
!/work2/07880/devansh/lonestar/software/plink \
  --bfile PRS_INPUT_FILES/subset_data \
  --clump PRS_INPUT_FILES/prs_input_all_SNPs_EUR.txt \
  --clump-snp-field SNP \
  --clump-p1 1 \
  --clump-p2 1 \
  --clump-r2 0.1 \
  --clump-kb 250 \
  --clump-field P \
  --out PRS_INPUT_FILES/meta_clumped_all_SNPs_EUR

In [ ]:
!awk 'NR > 1 {print $3}' PRS_INPUT_FILES/meta_clumped_all_SNPs_EUR.clumped > PRS_INPUT_FILES/clumped_snps_all_EUR.txt

In [ ]:
!grep -Fwf PRS_INPUT_FILES/clumped_snps_all_EUR.txt PRS_INPUT_FILES/prs_input_all_SNPs_EUR.txt > PRS_INPUT_FILES/prs_clumped_all_EUR_sumstats.txt

In [ ]:
# !awk '!seen[$1]++' PRS_INPUT_FILES/prs_clumped_3study_sumstats.txt > PRS_INPUT_FILES/prs_clumped_3study_sumstats_nodup.txt

In [ ]:
# !/work2/07880/devansh/lonestar/software/plink2 \
#   --bfile PRS_INPUT_FILES/subset_data \
#   --score PRS_INPUT_FILES/prs_clumped_3study_sumstats.txt 1 2 4 \
#   --pheno PRS_INPUT_FILES/subset_ids_plink.txt \
#   --pheno-name CAD \
#   --out PRS_INPUT_FILES/PRS_clumped_results

In [ ]:
pvalue_thresholds = [5e-8, 1e-6, 1e-4, 1e-2, 0.05, 0.1, 0.5, 1.0]

In [ ]:
import pandas as pd
import subprocess
import os

# Setup
sumstats_file = "PRS_INPUT_FILES/prs_clumped_all_EUR_sumstats.txt"
geno_prefix = "PRS_INPUT_FILES/subset_data"
pheno_file = "PRS_INPUT_FILES/subset_ids_plink.txt"
out_dir = "PRS_INPUT_FILES/PRS_THRESHOLDS"
os.makedirs(out_dir, exist_ok=True)

# Updated header order
sumstats = pd.read_csv(sumstats_file, sep="\t", header=None,
                       names=["SNP", "A1", "A2", "BETA", "P"])

sumstats['P'] = sumstats['P'].astype('float')
# Define p-value thresholds
thresholds = [ 0.5, 1.0] #5e-8, 1e-6, 1e-4, 1e-2, 0.05, 0.1,

# Run PRS at each threshold
for pval in thresholds:
    tag = str(pval).replace('.', 'p')
    filtered = sumstats[sumstats["P"] <= pval][["SNP", "A1", "BETA"]]
    score_file = f"{out_dir}/all_EUR_score_p{tag}.txt"
    out_prefix = f"{out_dir}/all_EUR_prs_p{tag}"

    # Save score file without header
    filtered.to_csv(score_file, sep="\t", index=False, header=False)

    cmd = [
        "/work2/07880/devansh/lonestar/software/plink2",
        "--bfile", geno_prefix,
        "--score", score_file, "1", "2", "3", "no-mean-imputation",
        "--pheno", pheno_file,
        "--pheno-name", "CAD",
        "--out", out_prefix
    ]

    print(f"Running PRS for P<={pval}, SNPs used: {filtered.shape[0]}")
    subprocess.run(cmd)

In [ ]:
thresholds = [5e-8, 1e-6, 1e-4, 1e-2, 0.05, 0.1, 0.5, 1.0]
for pval in thresholds:
    print(str(pval).replace('.', 'p'))

In [ ]:
# from scipy.stats import norm
# import os
# import glob
# import re
# import numpy as np
# import pandas as pd
# from sklearn.metrics import roc_auc_score
# import statsmodels.api as sm
# import matplotlib.pyplot as plt
# %matplotlib inline


# # SETTINGS
# SCORE_COLUMN = "SCORE1_AVG"
# PHENOTYPE_COLUMN = "CAD"
# PREVALENCE = 0.118  # CAD population prevalence
# SCORE_FOLDER = "PRS_INPUT_FILES/PRS_THRESHOLDS"

# # LIABILITY R2 conversion
# def compute_liability_r2(obs_r2, K, P):
#     """Convert observed Nagelkerke R² to liability scale R²"""
#     z = norm.ppf(1 - K)
#     phi = np.exp(-0.5 * z**2) / np.sqrt(2 * np.pi)
#     c = K * (1 - K) / (phi**2)
#     return obs_r2 * c / (P * (1 - P))

# def nagelkerke_r2(y, X):
#     X = sm.add_constant(X)
#     model = sm.Logit(y, X).fit(disp=0)
#     llf = model.llf
#     llnull = sm.Logit(y, sm.add_constant(np.ones(len(y)))).fit(disp=0).llf
#     return 1 - (llf / llnull)

# def parse_threshold(filename):
#     base = os.path.basename(filename).replace("all_EUR_prs_p", "").replace(".sscore", "")
#     if base == "1p0":
#         return 1.0
#     if "e" in base.lower():
#         return float(base.replace("p", "e-"))
#     if base.startswith("0p"):
#         return float("0." + base.split("p")[-1])
#     raise ValueError(f"Could not parse threshold from {base}")

# results = []

# for file in sorted(glob.glob(f"{SCORE_FOLDER}/all_EUR_prs_*.sscore")):
#     try:
#         threshold = parse_threshold(file)

#         df = pd.read_csv(file, delim_whitespace=True, comment="#",
#                          names=["FID", "IID", "CAD", "ALLELE_CT", "NAMED_ALLELE_DOSAGE_SUM", "SCORE1_AVG"])
#         if df.empty or df[SCORE_COLUMN].isnull().all():
#             print(f"⚠️ Skipping {file}: empty or missing scores")
#             continue

#         y = df[PHENOTYPE_COLUMN].replace({1: 0, 2: 1})
#         X = df[SCORE_COLUMN]

#         if y.nunique() < 2:
#             print(f"⚠️ Skipping {file}: only one phenotype class")
#             continue

#         auc = roc_auc_score(y, X)
#         nk_r2 = nagelkerke_r2(y, X)
#         sample_K = y.mean()
#         liability_r2 = compute_liability_r2(nk_r2, PREVALENCE, sample_K)

#         results.append({
#             "Threshold": threshold,
#             "AUC": auc,
#             "Nagelkerke_R2": nk_r2,
#             "Liability_R2": liability_r2
#         })

#     except Exception as e:
#         print(f"❌ Failed on {file}: {e}")

# # Display
# if results:
#     results_df = pd.DataFrame(results)
#     results_df = results_df.sort_values(by="Threshold")

#     # Remove highly negative R2 for visualization
#     results_df_filtered = results_df[results_df["Nagelkerke_R2"] > -1]

#     print("\n🔍 PRS Performance Summary:")
#     print(results_df_filtered[["Threshold", "AUC", "Nagelkerke_R2", "Liability_R2"]])

#     # Plot
#     plt.figure(figsize=(12, 5))

#     plt.subplot(1, 2, 1)
#     plt.plot(results_df_filtered["Threshold"], results_df_filtered["AUC"], marker='o')
#     plt.xscale('log')
#     plt.xlabel("P-value Threshold")
#     plt.ylabel("AUC")
#     plt.title("AUC vs P-value Threshold")

#     plt.subplot(1, 2, 2)
#     plt.plot(results_df_filtered["Threshold"], results_df_filtered["Liability_R2"], marker='o', label="Liability R²")
#     plt.plot(results_df_filtered["Threshold"], results_df_filtered["Nagelkerke_R2"], marker='s', label="Nagelkerke R²")
#     plt.xscale('log')
#     plt.xlabel("P-value Threshold")
#     plt.ylabel("R²")
#     plt.title("R² vs P-value Threshold")
#     plt.legend()

#     plt.tight_layout()
#     plt.show()
# else:
#     print("❌ No valid PRS results were found.")

In [ ]:
from scipy.stats import norm
import os
import glob
import re
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import statsmodels.api as sm
import matplotlib.pyplot as plt
%matplotlib inline


# SETTINGS
SCORE_COLUMN = "SCORE1_AVG"
PHENOTYPE_COLUMN = "CAD"
PREVALENCE = 0.118  # CAD population prevalence
SCORE_FOLDER = "PRS_INPUT_FILES/PRS_THRESHOLDS"

# LIABILITY R2 conversion
def compute_liability_r2(obs_r2, K, P):
    """Convert observed Nagelkerke R² to liability scale R²"""
    z = norm.ppf(1 - K)
    phi = np.exp(-0.5 * z**2) / np.sqrt(2 * np.pi)
    c = K * (1 - K) / (phi**2)
    return obs_r2 * c / (P * (1 - P))

def nagelkerke_r2(y, X):
    X = sm.add_constant(X)
    model = sm.Logit(y, X).fit(disp=0)
    llf = model.llf
    llnull = sm.Logit(y, sm.add_constant(np.ones(len(y)))).fit(disp=0).llf
    return 1 - (llf / llnull)

def parse_threshold(filename):
    base = os.path.basename(filename).replace("EUR_2study_prs_p", "").replace(".sscore", "")
    if base == "1p0":
        return 1.0
    if "e" in base.lower():
        return float(base.replace("p", "e-"))
    if base.startswith("0p"):
        return float("0." + base.split("p")[-1])
    raise ValueError(f"Could not parse threshold from {base}")

results = []

for file in sorted(glob.glob(f"{SCORE_FOLDER}/EUR_2study_prs_*.sscore")):
    try:
        threshold = parse_threshold(file)

        df = pd.read_csv(file, delim_whitespace=True, comment="#",
                         names=["FID", "IID", "CAD", "ALLELE_CT", "NAMED_ALLELE_DOSAGE_SUM", "SCORE1_AVG"])
        if df.empty or df[SCORE_COLUMN].isnull().all():
            print(f"⚠️ Skipping {file}: empty or missing scores")
            continue

        y = df[PHENOTYPE_COLUMN].replace({1: 0, 2: 1})
        X = df[SCORE_COLUMN]

        if y.nunique() < 2:
            print(f"⚠️ Skipping {file}: only one phenotype class")
            continue

        auc = roc_auc_score(y, X)
        nk_r2 = nagelkerke_r2(y, X)
        sample_K = y.mean()
        liability_r2 = compute_liability_r2(nk_r2, PREVALENCE, sample_K)

        results.append({
            "Threshold": threshold,
            "AUC": auc,
            "Nagelkerke_R2": nk_r2,
            "Liability_R2": liability_r2
        })

    except Exception as e:
        print(f"❌ Failed on {file}: {e}")

# Display
if results:
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(by="Threshold")

    # Remove highly negative R2 for visualization
    results_df_filtered = results_df[results_df["Nagelkerke_R2"] > -1]

    print("\n🔍 PRS Performance Summary:")
    print(results_df_filtered[["Threshold", "AUC", "Nagelkerke_R2", "Liability_R2"]])

    # Plot
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(results_df_filtered["Threshold"], results_df_filtered["AUC"], marker='o')
    plt.xscale('log')
    plt.xlabel("P-value Threshold")
    plt.ylabel("AUC")
    plt.title("AUC vs P-value Threshold")

    plt.subplot(1, 2, 2)
    plt.plot(results_df_filtered["Threshold"], results_df_filtered["Liability_R2"], marker='o', label="Liability R²")
    plt.plot(results_df_filtered["Threshold"], results_df_filtered["Nagelkerke_R2"], marker='s', label="Nagelkerke R²")
    plt.xscale('log')
    plt.xlabel("P-value Threshold")
    plt.ylabel("R²")
    plt.title("R² vs P-value Threshold")
    plt.legend()

    plt.tight_layout()
    plt.show()
else:
    print("❌ No valid PRS results were found.")


In [ ]:
from scipy.stats import norm
import os
import glob
import re
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import statsmodels.api as sm
import matplotlib.pyplot as plt
%matplotlib inline

# SETTINGS
SCORE_COLUMN = "SCORE1_AVG"
PHENOTYPE_COLUMN = "CAD"
AGE_SEX_PATHS = {
    "age": "/scratch/07880/devansh/CAD/fid21003.csv",
    "sex": "/scratch/07880/devansh/CAD/fid31.csv"
}
SCORE_FOLDER = "PRS_INPUT_FILES/PRS_THRESHOLDS"

# Read age and sex files
age_df = pd.read_csv(AGE_SEX_PATHS["age"]).rename(columns={'eid': 'IID', '21003-2.0': 'age'})[['IID', 'age']]
sex_df = pd.read_csv(AGE_SEX_PATHS["sex"]).rename(columns={'eid': 'IID', '31-0.0': 'sex'})[['IID', 'sex']]

# LIABILITY R2 conversion

def compute_liability_r2(obs_r2, K, P):
    """Convert observed Nagelkerke R² to liability scale R²"""
    z = norm.ppf(1 - K)
    phi = np.exp(-0.5 * z**2) / np.sqrt(2 * np.pi)
    c = K * (1 - K) / (phi**2)
    return obs_r2 * c / (P * (1 - P))

def nagelkerke_r2(y, X):
    X = sm.add_constant(X)
    model = sm.Logit(y, X).fit(disp=0)
    llf = model.llf
    llnull = sm.Logit(y, sm.add_constant(np.ones(len(y)))).fit(disp=0).llf
    return 1 - (llf / llnull)

def parse_threshold(filename):
    base = os.path.basename(filename).replace("prs_p", "").replace(".sscore", "")
    if base == "1p0":
        return 1.0
    if "e" in base.lower():
        return float(base.replace("p", "e-"))
    if base.startswith("0p"):
        return float("0." + base.split("p")[-1])
    raise ValueError(f"Could not parse threshold from {base}")

results = []

for file in sorted(glob.glob(f"{SCORE_FOLDER}/prs_*.sscore")):
    try:
        threshold = parse_threshold(file)

        df = pd.read_csv(file, delim_whitespace=True, comment="#",
                         names=["FID", "IID", "CAD", "ALLELE_CT", "NAMED_ALLELE_DOSAGE_SUM", "SCORE1_AVG"])
        if df.empty or df[SCORE_COLUMN].isnull().all():
            print(f"⚠️ Skipping {file}: empty or missing scores")
            continue

        # Merge age and sex
        df = df.merge(age_df, on="IID", how="inner").merge(sex_df, on="IID", how="inner")
        df = df.dropna(subset=["age", "sex", "SCORE1_AVG", "CAD"])

        # Encode phenotype
        y = df[PHENOTYPE_COLUMN].replace({1: 0, 2: 1})  # 1=control, 2=case
        if y.nunique() < 2:
            print(f"⚠️ Skipping {file}: only one phenotype class")
            continue

        # Covariates: age, sex, PRS
        X = df[[SCORE_COLUMN, "age", "sex"]]
        auc = roc_auc_score(y, df[SCORE_COLUMN])

        nk_r2 = nagelkerke_r2(y, X)
        sample_K = y.mean()  # Sample prevalence
        liability_r2 = compute_liability_r2(nk_r2, 0.118, sample_K)

        results.append({
            "Threshold": threshold,
            "Sample_Prevalence": sample_K,
            "AUC": auc,
            "Nagelkerke_R2": nk_r2,
            "Liability_R2": liability_r2
        })

    except Exception as e:
        print(f"❌ Failed on {file}: {e}")

# Display
if results:
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(by="Threshold")

    # Remove highly negative R2 for visualization
    results_df_filtered = results_df[results_df["Nagelkerke_R2"] > -1]

    print("\n🔍 PRS Performance Summary:")
    print(results_df_filtered[["Threshold", "AUC", "Nagelkerke_R2", "Liability_R2" ]])

    # Plot
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(results_df_filtered["Threshold"], results_df_filtered["AUC"], marker='o')
    plt.xscale('log')
    plt.xlabel("P-value Threshold")
    plt.ylabel("AUC")
    plt.title("AUC vs P-value Threshold")

    plt.subplot(1, 2, 2)
    plt.plot(results_df_filtered["Threshold"], results_df_filtered["Liability_R2"], marker='o', label="Liability R²")
    plt.plot(results_df_filtered["Threshold"], results_df_filtered["Nagelkerke_R2"], marker='s', label="Nagelkerke R²")
    plt.xscale('log')
    plt.xlabel("P-value Threshold")
    plt.ylabel("R²")
    plt.title("R² vs P-value Threshold")
    plt.legend()

    plt.tight_layout()
    plt.show()
else:
    print("❌ No valid PRS results were found.")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score
from scipy.stats import norm

# -------------------------------
# 🔧 Paths and Settings
# -------------------------------
PRS_FILE = "/scratch/07880/devansh/CAD/New_PRS/PRS_INPUT_FILES/PRS_THRESHOLDS/prs_p1p0.sscore"
PHENO_FILE = "/scratch/07880/devansh/CAD/New_PRS/PRS_INPUT_FILES/subset_ids_plink.txt"
AGE_FILE = "/scratch/07880/devansh/CAD/fid21003.csv"
SEX_FILE = "/scratch/07880/devansh/CAD/fid31.csv"
PREVALENCE = 0.0773

# -------------------------------
# 📥 Load Data
# -------------------------------
# PRS
prs = pd.read_csv(PRS_FILE, delim_whitespace=True)
prs = prs.drop(['CAD'], axis = 1)


# Phenotype: 1 = control, 2 = case → convert to binary: 0 = control, 1 = case
pheno = pd.read_csv(PHENO_FILE, delim_whitespace=True)
pheno["CAD"] = pheno["CAD"].replace({1: 0, 2: 1})


# Age
age = pd.read_csv(AGE_FILE).rename(columns={"eid": "IID", "21003-2.0": "age"})[["IID", "age"]]

# Sex
sex = pd.read_csv(SEX_FILE).rename(columns={"eid": "IID", "31-0.0": "sex"})[["IID", "sex"]]

# -------------------------------
# 🧬 Merge all data
# -------------------------------
df = prs.merge(pheno, on="IID")
df = df.merge(age, on="IID")
df = df.merge(sex, on="IID")

df = df.dropna()

print(df.head())
# -------------------------------
# 📈 Metrics
# -------------------------------
def compute_liability_r2(obs_r2, K, P):
    """Convert observed Nagelkerke R² to liability scale R²"""
    z = norm.ppf(1 - K)
    phi = np.exp(-0.5 * z**2) / np.sqrt(2 * np.pi)
    c = K * (1 - K) / (phi**2)
    return obs_r2 * c / (P * (1 - P))

def nagelkerke_r2(y, X):
    X = sm.add_constant(X)
    model = sm.Logit(y, X).fit(disp=0)
    llf = model.llf
    llnull = sm.Logit(y, sm.add_constant(np.ones(len(y)))).fit(disp=0).llf
    return 1 - (llf / llnull)


# -------------------------------
# 🧮 Compute
# -------------------------------
y = df["CAD"]
X_full = df[["SCORE1_AVG", "age", "sex"]]

# AUC
auc = roc_auc_score(y, df["SCORE1_AVG"])

# Nagelkerke R²
nk_r2 = nagelkerke_r2(y, X_full)

# Liability R²
sample_prev = y.mean()
liab_r2 = compute_liability_r2(nk_r2, PREVALENCE, sample_prev)

# -------------------------------
# 📊 Display
# -------------------------------
print("\n📍 PRS based on MetaGWAS on PRSsc with Age and Sex as Covariates")
print(f"Sample prevalence: {sample_prev:.3f}")
print(f"AUC:               {auc:.4f}")
print(f"Nagelkerke R²:     {nk_r2:.4f}")

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from sklearn.metrics import roc_auc_score
import statsmodels.api as sm

# -------------------------------
# 🔧 Paths and Settings
# -------------------------------
PRS_FILE = "/scratch/07880/devansh/CAD/Code_Analysis/prs_output_CAD_PRS_new.txt"
PHENO_FILE = "/scratch/07880/devansh/CAD/New_PRS/PRS_INPUT_FILES/subset_ids_plink.txt"
AGE_FILE = "/scratch/07880/devansh/CAD/fid21003.csv"
SEX_FILE = "/scratch/07880/devansh/CAD/fid31.csv"
PREVALENCE = 0.0773

# -------------------------------
# 📥 Load Data
# -------------------------------
# PRS
prs = pd.read_csv(PRS_FILE, delim_whitespace=True)

# Phenotype: 1 = control, 2 = case → convert to 0/1
pheno = pd.read_csv(PHENO_FILE, delim_whitespace=True)
pheno["CAD"] = pheno["CAD"].replace({1: 0, 2: 1})

# Age
age = pd.read_csv(AGE_FILE).rename(columns={"eid": "IID", "21003-2.0": "age"})[["IID", "age"]]

# Sex
sex = pd.read_csv(SEX_FILE).rename(columns={"eid": "IID", "31-0.0": "sex"})[["IID", "sex"]]

# -------------------------------
# 🧬 Merge all data
# -------------------------------
df = prs.merge(pheno, on="IID")
df = df.merge(age, on="IID")
df = df.merge(sex, on="IID")
df = df.dropna()

# -------------------------------
# 📈 Metric Functions
# -------------------------------
def compute_liability_r2(obs_r2, K, P):
    """Convert observed Nagelkerke R² to liability scale R²"""
    z = norm.ppf(1 - K)
    phi = np.exp(-0.5 * z**2) / np.sqrt(2 * np.pi)
    c = K * (1 - K) / (phi**2)
    return obs_r2 * c / (P * (1 - P))

def nagelkerke_r2(y, X):
    X = sm.add_constant(X)
    model = sm.Logit(y, X).fit(disp=0)
    llf = model.llf
    llnull = sm.Logit(y, sm.add_constant(np.ones(len(y)))).fit(disp=0).llf
    return 1 - (llf / llnull)

# -------------------------------
# 🧮 Compute Scores
# -------------------------------
y = df["CAD"]
X_cov = df[["SCORE1_SUM", "age", "sex"]]

# AUC
auc = roc_auc_score(y, df["SCORE1_SUM"])

# R²
nk_r2 = nagelkerke_r2(y, X_cov)
sample_prev = y.mean()
liab_r2 = compute_liability_r2(nk_r2, PREVALENCE, sample_prev)

# -------------------------------
# 📊 Output
# -------------------------------
print("\n📍 GPSmult (with age + sex as covariates)")
print(f"Sample prevalence: {sample_prev:.3f}")
print(f"AUC:               {auc:.4f}")
print(f"Nagelkerke R²:     {nk_r2:.4f}")

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from sklearn.metrics import roc_auc_score
import statsmodels.api as sm

# -------------------------------
# 🔧 Paths and Settings
# -------------------------------
PRS_FILE = "/scratch/07880/devansh/CAD/New_PRS/PRS_INPUT_FILES/PRS_THRESHOLDS/prs_output_CAD_old_regenie.txt"
PHENO_FILE = "/scratch/07880/devansh/CAD/New_PRS/PRS_INPUT_FILES/subset_ids_plink.txt"
AGE_FILE = "/scratch/07880/devansh/CAD/fid21003.csv"
SEX_FILE = "/scratch/07880/devansh/CAD/fid31.csv"
PREVALENCE = 0.0773

# -------------------------------
# 📥 Load Data
# -------------------------------
# PRS
prs = pd.read_csv(PRS_FILE, delim_whitespace=True)

# Phenotype: 1 = control, 2 = case → convert to 0/1
pheno = pd.read_csv(PHENO_FILE, delim_whitespace=True)
pheno["CAD"] = pheno["CAD"].replace({1: 0, 2: 1})

# Age
age = pd.read_csv(AGE_FILE).rename(columns={"eid": "IID", "21003-2.0": "age"})[["IID", "age"]]

# Sex
sex = pd.read_csv(SEX_FILE).rename(columns={"eid": "IID", "31-0.0": "sex"})[["IID", "sex"]]

# -------------------------------
# 🧬 Merge all data
# -------------------------------
df = prs.merge(pheno, on="IID")
df = df.merge(age, on="IID")
df = df.merge(sex, on="IID")
df = df.dropna()

# -------------------------------
# 📈 Metric Functions
# -------------------------------
def compute_liability_r2(obs_r2, K, P):
    """Convert observed Nagelkerke R² to liability scale R²"""
    z = norm.ppf(1 - K)
    phi = np.exp(-0.5 * z**2) / np.sqrt(2 * np.pi)
    c = K * (1 - K) / (phi**2)
    return obs_r2 * c / (P * (1 - P))

def nagelkerke_r2(y, X):
    X = sm.add_constant(X)
    model = sm.Logit(y, X).fit(disp=0)
    llf = model.llf
    llnull = sm.Logit(y, sm.add_constant(np.ones(len(y)))).fit(disp=0).llf
    return 1 - (llf / llnull)

# -------------------------------
# 🧮 Compute Scores
# -------------------------------
y = df["CAD"]
X_cov = df[["SCORE1_SUM", "age", "sex"]]

# AUC
auc = roc_auc_score(y, df["SCORE1_SUM"])

# R²
nk_r2 = nagelkerke_r2(y, X_cov)
sample_prev = y.mean()
liab_r2 = compute_liability_r2(nk_r2, PREVALENCE, sample_prev)

# -------------------------------
# 📊 Output
# -------------------------------
print("\n📍 UKB Case Control PRS (with age + sex as covariates)")
print(f"Sample prevalence: {sample_prev:.3f}")
print(f"AUC:               {auc:.4f}")
print(f"Nagelkerke R²:     {nk_r2:.4f}")

In [ ]:
results_df_filtered[["Threshold", "AUC", "Nagelkerke_R2"]]

In [ ]:
df['label'].value_counts()

In [ ]:
df["PRS_scaled"] = (df["SCORE1_AVG"] - df["SCORE1_AVG"].mean()) / df["SCORE1_AVG"].std()

In [ ]:
import statsmodels.api as sm

X = df[["PRS_scaled"]]  # add covariates like 'age', 'sex', 'PC1' etc. if available
X = sm.add_constant(X)
y = df["label"]

model = sm.Logit(y, X).fit()
print(model.summary())

In [ ]:
import numpy as np
odds_ratio = np.exp(model.params["PRS_scaled"])
ci = np.exp(model.conf_int().loc["PRS_scaled"])
print(f"Odds Ratio: {odds_ratio:.2f} (95% CI: {ci[0]:.2f} – {ci[1]:.2f})")

In [ ]:
df["decile"] = pd.qcut(df["PRS_scaled"], 10, labels=False)
summary = df.groupby("decile")["label"].mean()

summary.plot(kind="bar", figsize=(8,5))
plt.ylabel("Proportion with CAD")
plt.xlabel("PRS Decile")
plt.title("CAD Prevalence Across PRS Deciles")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

# === Step 1: Load PRS results file ===
# Replace with your actual path
prs_path = "PRS_INPUT_FILES/PRS_results.sscore"

df = pd.read_csv(prs_path, delim_whitespace=True)

# Make sure columns are correctly labeled
assert {'CAD', 'SCORE1_AVG'}.issubset(df.columns), "Required columns missing."

# === Step 2: Standardize PRS ===
scaler = StandardScaler()
df["PRS_scaled"] = scaler.fit_transform(df[["SCORE1_AVG"]])

# === Step 3: Prepare variables for logistic regression ===
X = sm.add_constant(df["PRS_scaled"])
y = df["CAD"].replace({1: 0, 2: 1})  # PLINK uses 1=control, 2=case

# === Step 4: Fit logistic regression model ===
model = sm.Logit(y, X).fit(disp=False)

# === Step 5: Compute Nagelkerke R² ===
llf_full = model.llf
llf_null = model.llnull
n = model.nobs

R2_cs = 1 - np.exp((llf_null - llf_full) * 2 / n)
R2_nagelkerke = R2_cs / (1 - np.exp(llf_null * 2 / n))

print(f"\n📊 Nagelkerke R² = {R2_nagelkerke:.4f}")